In [1]:
import polars as pl
from pathlib import Path

In [2]:
DATA_GENERAL = Path("../data_general")
DATA_PERSONAL = Path("../data_personal/Spotify Extended Streaming History")

In [3]:
general_data_frames = []
for num in range (10) :
    general_data_frames.append(pl.read_parquet(DATA_GENERAL / f'spotify_audio_features_{num}.parquet'))

In [4]:
display(general_data_frames[0].head())

id,name,popularity,null_response,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2Pe9cbhOTvOUTDE4bl7zzl""","""I dreamt you died""",0,0,630506,4,6,0,87.683,0.279,0.391,-12.054,0.32,0.816,0.737,0.177,0.0299
"""0wP732NKm8XgXu78XLRWoR""","""It's Death""",0,0,97216,4,5,1,105.298,0.429,0.318,-11.685,0.0566,0.587,0.782,0.202,0.36
"""22L6EJdnjx8oIo7GiF9hLe""","""Preliminary""",0,0,75180,4,0,1,117.657,0.283,0.581,-9.42,0.0555,0.923,0.939,0.106,0.0362
"""3a519lgQ13JXNi0G73mwMT""","""Disparage""",0,0,149447,4,5,0,100.685,0.244,0.995,-0.69,0.125,0.78,0.799,0.132,0.0634
"""27yP7p2lxWYTtnldRN8Kzx""","""Cut Down""",0,0,120816,4,7,1,123.499,0.313,0.618,0.411,0.073,0.843,0.109,0.126,0.187


In [5]:
personal_data_frames = {}
for num in range (2022,2027) :
    personal_data_frames[num] = pl.read_json(
        DATA_PERSONAL / f'Streaming_History_Audio_{num}.json',
        infer_schema_length=None  
        )

In [6]:
personal_data_frames[2022] =personal_data_frames[2022].with_columns(pl.col("spotify_track_uri").str.slice(14,22))

In [7]:
check_mus_id = personal_data_frames[2022]["spotify_track_uri"][0]
print(check_mus_id)

4u7EnebtmKWzUH433cf5Qv


In [8]:
#print(len(general_data_frames))
for g in range(10):
    general_data_frames[g] = general_data_frames[g].rename({ 'id' : 'spotify_track_uri' })
print(general_data_frames[0])

shape: (25_558_893, 17)
┌────────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬──────────┬─────────┐
│ spotify_tr ┆ name       ┆ popularit ┆ null_resp ┆ … ┆ acousticn ┆ instrumen ┆ liveness ┆ valence │
│ ack_uri    ┆ ---        ┆ y         ┆ onse      ┆   ┆ ess       ┆ talness   ┆ ---      ┆ ---     │
│ ---        ┆ str        ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ f64      ┆ f64     │
│ str        ┆            ┆ i64       ┆ i64       ┆   ┆ f64       ┆ f64       ┆          ┆         │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪══════════╪═════════╡
│ 2Pe9cbhOTv ┆ I dreamt   ┆ 0         ┆ 0         ┆ … ┆ 0.816     ┆ 0.737     ┆ 0.177    ┆ 0.0299  │
│ OUTDE4bl7z ┆ you died   ┆           ┆           ┆   ┆           ┆           ┆          ┆         │
│ zl         ┆            ┆           ┆           ┆   ┆           ┆           ┆          ┆         │
│ 0wP732NKm8 ┆ It's Death ┆ 0         ┆ 0         ┆ … ┆ 0.587     ┆

In [ ]:
status = False
mus_name = ''
for g in range(10):
    for url in general_data_frames[g]['spotify_track_uri']:
        if url == check_mus_id:
            status = True
            mus_name = general_data_frames[g].filter(pl.col('spotify_track_uri') == url).select(['name',])
            break
    if status :
        break

print(status, mus_name)

True shape: (1, 1)
┌─────────────────────────────────┐
│ name                            │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ Bohemian Rhapsody - Remastered… │
└─────────────────────────────────┘


# NOW LET`S FIND OUT TRACK

In [12]:
song_uri = '3U0UXxBIfjUsJ8RtxoxUFn'

In [28]:

for g in range (10):
    if song_uri in general_data_frames[g]['spotify_track_uri']:
        print(g)
        stats = general_data_frames[g].filter(pl.col('spotify_track_uri') == song_uri).select(['name','tempo','danceability','energy','loudness','speechiness','acousticness','instrumentalness','liveness','valence'])
        break


2


In [16]:
display(stats)

name,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Monolith""",142.013,0.508,0.952,-6.445,0.0405,0.000409,0.756,0.114,0.799


# I have priority on the  energy, instrumentalness and valence

In [ ]:
energy = (float(stats['energy'][0]) - 0.02 , float(stats['energy'][0]) + 0.02 )
instr = (float(stats['instrumentalness'][0]) - 0.02 , float(stats['instrumentalness'][0]) + 0.02 )
valence = (float(stats['valence'][0]) - 0.02 , float(stats['valence'][0]) + 0.02 )

print(energy)

(0.9319999999999999, 0.972)


In [29]:
result = general_data_frames[2].filter((pl.col('energy').is_between(energy[0], energy[1])) & 
                                        (pl.col('instrumentalness').is_between(instr[0], instr[1])) &
                                        (pl.col('valence').is_between(valence[0], valence[1]))
                                        )

In [30]:
display(result)

spotify_track_uri,name,popularity,null_response,duration_ms,time_signature,key,mode,tempo,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
str,str,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""04A1SwAIRop8g5m26Ev9l4""","""High Speed""",0,0,146667,4,0,1,126.013,0.582,0.946,-5.904,0.0428,0.146,0.764,0.171,0.779
"""429WhcRrxHnUXmhwcuRiKd""","""Drugs""",0,0,138500,4,7,1,120.011,0.821,0.936,-4.594,0.0648,0.00132,0.754,0.0549,0.781
"""5QJkaoDwDn6t7FByv9XDP5""","""Sunset Overdrive""",0,0,212120,4,1,0,119.974,0.736,0.932,-4.587,0.0371,0.0942,0.751,0.0925,0.786
"""2mAPoAdGe6Bdo92ZC0i29D""","""Needle Nose""",0,0,147800,4,6,0,146.333,0.486,0.945,-9.083,0.0531,0.0144,0.776,0.397,0.779
"""7fR7Lxi5ZPbZjsiKk5kgqD""","""JING JANAI JONG KA JINGIEID""",2,0,282816,4,9,1,137.992,0.787,0.946,-4.675,0.0346,0.000351,0.739,0.0801,0.795
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""5uJjwl8WF1nSPiNNye3n80""","""Alley's Ode""",0,0,174760,4,6,0,181.97,0.32,0.949,-7.752,0.041,0.00271,0.755,0.24,0.795
"""5OY9wvkAIjTQb3W2AUEJ5k""","""Get Your Money""",0,0,269589,4,1,1,146.036,0.741,0.947,-6.303,0.0475,0.000264,0.762,0.342,0.802
"""1aAoAfhXL8Hgnj9JklKkwl""","""The Dance That Never Ends""",0,0,126622,4,9,0,165.245,0.669,0.94,-6.714,0.042,0.1,0.757,0.211,0.799
